In [1]:
import torch

import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from PIL import Image

import random
random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = Path("/kaggle/input/sticky-note-blindness-aicc-round-4")
BATCH_SIZE = 64

In [2]:
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
print(f"CLIP ViT-B/16 loaded on {DEVICE}")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP ViT-B/16 loaded on cuda


In [3]:

text_embs = torch.load(DATA / "class_embeddings.pt", weights_only=True).to(DEVICE)
text_embs = F.normalize(text_embs.float(), dim=-1)
metadata = pd.read_csv(DATA / "metadata.csv")
N, K = len(metadata), text_embs.shape[0]
valid_ids = set(metadata["sample_id"].tolist())

all_paths = sorted((DATA / "images" / "images").glob("*.png"), key=lambda p: int(p.stem))
image_paths = [p for p in all_paths if int(p.stem) in valid_ids]
print(f"Images: {len(image_paths)}  |  Classes: {K}  |  Embedding dim: {text_embs.shape[1]}")

Images: 348  |  Classes: 764  |  Embedding dim: 512


In [24]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image

@torch.inference_mode()
def encode_images2(paths, batch_size=BATCH_SIZE):
    all_embs = []
    for i in tqdm(range(0, len(paths), batch_size), desc="Encoding"):
        batch = [Image.open(p).convert("RGB").transpose(Image.FLIP_LEFT_RIGHT) for p in paths[i:i + batch_size]]
        inputs = processor(images=batch, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)

        vision_out = model.vision_model(pixel_values=pixel_values)
        pooled = vision_out.pooler_output
        if pooled is None:
            pooled = vision_out.last_hidden_state[:, 0, :]
        emb = model.visual_projection(pooled)

        batch2 = [Image.open(p).convert("RGB") for p in paths[i:i+batch_size]]
        inputs2 = processor(images=batch2, return_tensors='pt')
        pixel_values2 = inputs2['pixel_values'].to(DEVICE)
        
        vision_out2 = model.vision_model(pixel_values=pixel_values2)
        pooled2 = vision_out2.pooler_output 
        if pooled2 is None:
            pooled2 = vision_out2.last_hidden_state[:, 0, :] 
        emb2 = model.visual_projection(pooled2)
        
        emb = F.normalize(emb.float(), dim=-1)
        emb2 = F.normalize(emb2.float(), dim=-1)
        
        final = (emb * emb2) > 0
        
        emb_final = torch.where(final, emb + emb2, emb) 
        
        emb_final = F.normalize(emb_final, dim=-1)
        all_embs.append(emb_final.cpu())

    return torch.cat(all_embs, dim=0)

In [27]:
image_embs = encode_images2(image_paths).to(DEVICE)
predictions = (image_embs @ text_embs.T).argmax(dim=1).cpu().tolist()
print(f"Generated {len(predictions)} predictions.")

Encoding: 100%|██████████| 6/6 [00:12<00:00,  2.03s/it]

Generated 348 predictions.


In [28]:
submission = pd.DataFrame({"sample_id": metadata["sample_id"].tolist(), "label": predictions})
submission.to_csv("submission.csv", index=False)
print(f"Submission shape: {submission.shape}")
submission.head()

Submission shape: (348, 2)


,sample_id,label
0,0,237
1,1,355
2,2,13
3,3,693
4,4,164
